In [1]:
# ==================== FUNCTION DEFINITION =====================

def process_fsc_data(coords: [float, float, float, float],
                     selected_dates: list,
                     output_folder: str,
                     time_interval: [str, str],
                     eval_fsc_scl_mask,
                     glacier_shp: str,
                     lake_shp: str,
                     forest_mask_path: str,
                     resolution):

    import os
    import shutil
    from pathlib import Path
    from datetime import datetime, timedelta

    import numpy as np
    import pandas as pd
    import geopandas as gpd
    import rioxarray as rxr
    import xarray as xr
    import re

    from rasterio.features import rasterize
    from sentinelhub import (
        SHConfig, DataCollection, SentinelHubCatalog, SentinelHubRequest,
        MimeType, BBox, bbox_to_dimensions, CRS
    )

    config = SHConfig("MO_v2")
    if not config.sh_client_id or not config.sh_client_secret:
        raise RuntimeError("SentinelHub credentials missing!")

    epsg = 32632
    aoi_bbox = BBox(bbox=coords, crs=CRS(epsg))
    aoi_size = bbox_to_dimensions(aoi_bbox, resolution=resolution)
    print(f"Image shape at {resolution} m resolution: {aoi_size} pixels")

    catalog = SentinelHubCatalog(config=config)
    search_iterator = catalog.search(
        DataCollection.SENTINEL2_L2A,
        bbox=aoi_bbox,
        time=time_interval,
        fields={"include": ["id", "properties.datetime"], "exclude": []},
        filter="eo:cloud_cover < 50"
    )

    results = list(search_iterator)
    print("Total number of results:", len(results))

    datetime_list = sorted({datetime.strptime(result["properties"]["datetime"][:10], '%Y-%m-%d') for result in results})
    print("Available image dates:", datetime_list)

    # Download images
    ndsi_images = []
    for current_date in datetime_list:
        time_interval_day = (current_date.strftime("%Y-%m-%d"), current_date.strftime("%Y-%m-%d"))
        print(f"Downloading for {time_interval_day[0]}")
        request_ndsi = SentinelHubRequest(
            data_folder=(f"{output_folder}/{time_interval_day[0]}"),
            evalscript=eval_fsc_scl_mask,
            input_data=[
                SentinelHubRequest.input_data(
                    data_collection=DataCollection.SENTINEL2_L2A.define_from("s2l2a", service_url=config.sh_base_url),
                    time_interval=time_interval_day
                )
            ],
            responses=[SentinelHubRequest.output_response("default", MimeType.TIFF)],
            bbox=aoi_bbox,
            size=aoi_size,
            config=config,
        )
        ndsi_imgs = request_ndsi.get_data(save_data=True)
        ndsi_images.extend(ndsi_imgs)

    # Reorganize folder structure
    def process_tiff_files(base_path):
        base_path = Path(base_path)
        for date_folder in base_path.iterdir():
            if date_folder.is_dir():
                nested = list(date_folder.glob("*/response.tiff"))
                if len(nested) == 1:
                    new_name = f"ndsi_{date_folder.name}.tiff"
                    shutil.move(str(nested[0]), base_path / new_name)
                    shutil.rmtree(date_folder)
                    print(f"Moved and renamed {new_name}")
        print("Folder structure reorganized.")

    process_tiff_files(output_folder)

    # Combine datasets
    pattern = re.compile(r"ndsi_(\d{4}-\d{2}-\d{2})\.tiff")
    datasets = []
    for file in os.listdir(output_folder):
        if pattern.match(file):
            date_str = pattern.match(file).group(1)
            date = pd.to_datetime(date_str).date()
            dataset = rxr.open_rasterio(os.path.join(output_folder, file)).expand_dims(time=[pd.Timestamp(date)])
            datasets.append(dataset)

    if not datasets:
        print("No datasets found!")
        return

    combined_dataset = xr.concat(datasets, dim="time")
    combined_dataset.rio.write_nodata(np.nan, inplace=True)

    # Filter based on NaN coverage
    nan_fraction = combined_dataset.isnull().sum(dim=["x", "y"]) / (combined_dataset.sizes["x"] * combined_dataset.sizes["y"])
    valid_indices = (nan_fraction <= 0.70).values.flatten()
    combined_dataset = combined_dataset.isel(time=valid_indices)

    print(f"Filtered dataset shape: {combined_dataset.shape}")

    if not selected_dates:
        print("No selected dates provided. Skipping download and processing.")
        return

    # Copy selected date images
    selected_dir = os.path.join(output_folder, "selected")
    os.makedirs(selected_dir, exist_ok=True)
    selected_date_strs = set(dt.strftime("%Y-%m-%d") for dt in selected_dates)

    for file in os.listdir(output_folder):
        if file.startswith("ndsi_") and file.endswith(".tiff"):
            if file[5:-5] in selected_date_strs:
                shutil.copy2(os.path.join(output_folder, file), os.path.join(selected_dir, file))
                print(f"Copied {file} to selected folder")

    # Apply glacier, lake, and forest masks
    noforest_dir = os.path.join(output_folder, "selected_masked")
    os.makedirs(noforest_dir, exist_ok=True)

    glacier_gdf = gpd.read_file(glacier_shp)
    lake_gdf = gpd.read_file(lake_shp)
    forest_mask = rxr.open_rasterio(forest_mask_path).squeeze()

    for file in os.listdir(selected_dir):
        if file.endswith(".tiff") and file.startswith("ndsi_"):
            img = rxr.open_rasterio(os.path.join(selected_dir, file)).squeeze()
            forest_mask_matched = forest_mask.rio.reproject_match(img)

            glacier_mask = rasterize(
                [(geom, 1) for geom in glacier_gdf.to_crs(img.rio.crs).geometry],
                out_shape=img.shape,
                transform=img.rio.transform(),
                fill=0,
                dtype="uint8"
            )

            lake_mask = rasterize(
                [(geom, 1) for geom in lake_gdf.to_crs(img.rio.crs).geometry],
                out_shape=img.shape,
                transform=img.rio.transform(),
                fill=0,
                dtype="uint8"
            )

            combined_mask = (forest_mask_matched.values == 1) | (glacier_mask == 1) | (lake_mask == 1)
            masked_img = img.where(~combined_mask)

            masked_img.rio.to_raster(os.path.join(noforest_dir, file))
            print(f"Masked and saved: {file}")

In [2]:
# define coordinates for the bbox
coords_achensee=(688675, 5250625, 710875, 5274625)
coords_kuehtai=(645975, 5204025, 673175, 5234725)
coords_kaunertal=(618775, 5187025, 646775, 5213325)

coords_engadin=(550975, 5131625,  613475, 5198125)
coords_ziller=(676275, 5205325, 738375, 5260925)
coords_nordost=(688675, 5239325, 749975, 5286925)
coords_zentral=(583375, 5180625,  701875, 5249625)

# Define all selected dates per AOI and season
from datetime import datetime

selected_dates_kuehtai_22_23 = [
    datetime(2022, 10, 3), datetime(2022, 10, 28), datetime(2023, 3, 22),
    datetime(2023, 6, 7), datetime(2023, 6, 20), datetime(2023, 6, 25),
    datetime(2023, 7, 15), datetime(2023, 8, 11)
]

selected_dates_kuehtai_23_24 = [
    datetime(2023, 10, 13), datetime(2023, 10, 28), datetime(2024, 2, 5),
    datetime(2024, 3, 11), datetime(2024, 4, 12), datetime(2024, 5, 20),
    datetime(2024, 6, 19), datetime(2024, 6, 29), datetime(2024, 7, 9),
    datetime(2024, 7, 14), datetime(2024, 8, 10)
]

selected_dates_kaunertal_22_23 = [
    datetime(2022, 10, 3), datetime(2022, 10, 28), datetime(2023, 3, 22),
    datetime(2023, 5, 3), datetime(2023, 6, 7), datetime(2023, 6, 25),
    datetime(2023, 7, 15), datetime(2023, 8, 11)
]

selected_dates_kaunertal_23_24 = [
    datetime(2023, 10, 25), datetime(2023, 10, 28), datetime(2024, 2, 5),
    datetime(2024, 3, 11), datetime(2024, 4, 12), datetime(2024, 5, 20),
    datetime(2024, 6, 19), datetime(2024, 6, 29), datetime(2024, 7, 9),
    datetime(2024, 7, 14), datetime(2024, 8, 10)
]

selected_dates_achensee_22_23 = [
    datetime(2022, 11, 9), datetime(2022, 11, 27), datetime(2023, 1, 1),
    datetime(2023, 2, 10), datetime(2023, 2, 15), datetime(2023, 2, 22),
    datetime(2023, 3, 7), datetime(2023, 3, 22), datetime(2023, 4, 21)
]

selected_dates_achensee_23_24 = [
    datetime(2023, 11, 4), datetime(2023, 12, 27), datetime(2024, 1, 3),
    datetime(2024, 2, 15), datetime(2024, 3, 8), datetime(2024, 4, 7),
    datetime(2024, 4, 27), datetime(2024, 4, 30)
]

# define the eval script
eval_fsc_scl_mask = """
//VERSION=3
function setup() {
  return {
    input: ["B03", "B11", "SCL"],
    output: { bands: 1, sampleType: "FLOAT32" }
  };
}

function evaluatePixel(sample) {
  let ndsi = (sample.B03 - sample.B11) / (sample.B03 + sample.B11);
  let image_mask = sample.SCL;

  // Mask values: adjust as needed
  let mask_values = [1,6,8,9];

  // FSC thresholds
  let ndsi_min = 0.1;
  let ndsi_max = 0.6;

  // Calculate FSC
  let fsc = 0.0;
  if (ndsi <= ndsi_min) {
    fsc = 0.0;
  } else if (ndsi >= ndsi_max) {
    fsc = 1.0;
  } else {
    fsc = (ndsi - ndsi_min) / (ndsi_max - ndsi_min);
  }

  // Mask out unwanted pixels
  let fsc_masked = mask_values.includes(image_mask) ? NaN : fsc;

  return [fsc_masked];
}
"""

### CALL FUNCTIONS!

In [3]:
# Example use for ziller 2018-2019
process_fsc_data(
    coords= coords_ziller,
    resolution=100,
    selected_dates= [],
    output_folder=r"C:\Users\Moritz\Documents\Uni\Masterarbeit\YETI_MA\Masterarbeit\data\FSC_data\ziller",
    time_interval=("2020-10-01", "2020-11-15"),
    eval_fsc_scl_mask=eval_fsc_scl_mask,
    glacier_shp = r"C:\Users\Moritz\Documents\Uni\Masterarbeit\YETI_MA\Masterarbeit\data\shapefiles\GI_5\Gletscher_AOIs.shp",
    lake_shp = r"C:\Users\Moritz\Documents\Uni\Masterarbeit\YETI_MA\Masterarbeit\data\shapefiles\Seen_AOIs.shp",
    forest_mask_path = r"C:\Users\Moritz\Documents\Uni\Masterarbeit\YETI_MA\Masterarbeit\data\ses_topo_22_forest_mask_domain_epsg32632.tif"
    )

Image shape at 100 m resolution: (621, 556) pixels
Total number of results: 13
Available image dates: [datetime.datetime(2020, 10, 8, 0, 0), datetime.datetime(2020, 10, 18, 0, 0), datetime.datetime(2020, 10, 25, 0, 0), datetime.datetime(2020, 11, 2, 0, 0), datetime.datetime(2020, 11, 7, 0, 0), datetime.datetime(2020, 11, 9, 0, 0), datetime.datetime(2020, 11, 12, 0, 0), datetime.datetime(2020, 11, 14, 0, 0)]
Moved and renamed ndsi_2020-10-08.tiff
Moved and renamed ndsi_2020-10-18.tiff
Moved and renamed ndsi_2020-10-25.tiff
Moved and renamed ndsi_2020-11-02.tiff
Moved and renamed ndsi_2020-11-07.tiff
Moved and renamed ndsi_2020-11-09.tiff
Moved and renamed ndsi_2020-11-12.tiff
Moved and renamed ndsi_2020-11-14.tiff
Folder structure reorganized.
Filtered dataset shape: (4, 1, 834, 931)
No selected dates provided. Skipping download and processing.


In [4]:
# Example use for Achensee 22-23
process_fsc_data(
    coords= coords_achensee,
    resolution=20,
    selected_dates= selected_dates_achensee_22_23,
    output_folder=r"C:\Users\Moritz\Documents\Uni\Masterarbeit\YETI_MA\Masterarbeit\data\FSC_data\Achensee_22-23",
    time_interval=("2022-10-01", "2023-06-30"),
    eval_fsc_scl_mask=eval_fsc_scl_mask,
    glacier_shp = r"C:\Users\Moritz\Documents\Uni\Masterarbeit\YETI_MA\Masterarbeit\data\shapefiles\GI_5\Gletscher_AOIs.shp",
    lake_shp = r"C:\Users\Moritz\Documents\Uni\Masterarbeit\YETI_MA\Masterarbeit\data\shapefiles\Seen_AOIs.shp",
    forest_mask_path = r"C:\Users\Moritz\Documents\Uni\Masterarbeit\YETI_MA\Masterarbeit\data\ses_topo_22_forest_mask_domain_epsg32632.tif"
    )

Image shape at 20 m resolution: (1110, 1200) pixels
Total number of results: 71
Available image dates: [datetime.datetime(2022, 10, 3, 0, 0), datetime.datetime(2022, 10, 5, 0, 0), datetime.datetime(2022, 10, 10, 0, 0), datetime.datetime(2022, 10, 18, 0, 0), datetime.datetime(2022, 10, 20, 0, 0), datetime.datetime(2022, 10, 25, 0, 0), datetime.datetime(2022, 10, 28, 0, 0), datetime.datetime(2022, 10, 30, 0, 0), datetime.datetime(2022, 11, 9, 0, 0), datetime.datetime(2022, 11, 14, 0, 0), datetime.datetime(2022, 11, 19, 0, 0), datetime.datetime(2022, 11, 27, 0, 0), datetime.datetime(2022, 12, 17, 0, 0), datetime.datetime(2022, 12, 22, 0, 0), datetime.datetime(2022, 12, 29, 0, 0), datetime.datetime(2023, 1, 1, 0, 0), datetime.datetime(2023, 1, 6, 0, 0), datetime.datetime(2023, 1, 16, 0, 0), datetime.datetime(2023, 1, 18, 0, 0), datetime.datetime(2023, 1, 26, 0, 0), datetime.datetime(2023, 1, 31, 0, 0), datetime.datetime(2023, 2, 5, 0, 0), datetime.datetime(2023, 2, 7, 0, 0), datetime.datet

In [ ]:
# Example use for Achensee 23-24
process_fsc_data(
    coords= coords_achensee,
    resolution=20,
    selected_dates= selected_dates_achensee_23_24,
    output_folder=r"C:\Users\Moritz\Documents\Uni\Masterarbeit\YETI_MA\Masterarbeit\data\FSC_data\Achensee_23-24",
    time_interval=("2022-10-01", "2023-06-30"),
    eval_fsc_scl_mask=eval_fsc_scl_mask,
    glacier_shp = r"C:\Users\Moritz\Documents\Uni\Masterarbeit\YETI_MA\Masterarbeit\data\shapefiles\GI_5\Gletscher_AOIs.shp",
    lake_shp = r"C:\Users\Moritz\Documents\Uni\Masterarbeit\YETI_MA\Masterarbeit\data\shapefiles\Seen_AOIs.shp",
    forest_mask_path = r"C:\Users\Moritz\Documents\Uni\Masterarbeit\YETI_MA\Masterarbeit\data\ses_topo_22_forest_mask_domain_epsg32632.tif"
    )

Image shape at 20 m resolution: (1110, 1200) pixels
Total number of results: 128
Available image dates: [datetime.datetime(2022, 10, 3, 0, 0), datetime.datetime(2022, 10, 5, 0, 0), datetime.datetime(2022, 10, 10, 0, 0), datetime.datetime(2022, 10, 18, 0, 0), datetime.datetime(2022, 10, 20, 0, 0), datetime.datetime(2022, 10, 25, 0, 0), datetime.datetime(2022, 10, 28, 0, 0), datetime.datetime(2022, 10, 30, 0, 0), datetime.datetime(2022, 11, 9, 0, 0), datetime.datetime(2022, 11, 14, 0, 0), datetime.datetime(2022, 11, 19, 0, 0), datetime.datetime(2022, 11, 27, 0, 0), datetime.datetime(2022, 12, 17, 0, 0), datetime.datetime(2022, 12, 22, 0, 0), datetime.datetime(2022, 12, 29, 0, 0), datetime.datetime(2023, 1, 1, 0, 0), datetime.datetime(2023, 1, 6, 0, 0), datetime.datetime(2023, 1, 16, 0, 0), datetime.datetime(2023, 1, 18, 0, 0), datetime.datetime(2023, 1, 26, 0, 0), datetime.datetime(2023, 1, 31, 0, 0), datetime.datetime(2023, 2, 5, 0, 0), datetime.datetime(2023, 2, 7, 0, 0), datetime.date

KeyboardInterrupt: 

In [6]:
# Example use for Kuehtai 22-23
process_fsc_data(
    coords= coords_kuehtai,
    resolution=20,
    selected_dates= selected_dates_kuehtai_22_23,
    output_folder=r"C:\Users\Moritz\Documents\Uni\Masterarbeit\YETI_MA\Masterarbeit\data\FSC_data\Kuehtai_22-23",
    time_interval=("2022-10-01", "2023-08-31"),
    eval_fsc_scl_mask=eval_fsc_scl_mask,
    glacier_shp = r"C:\Users\Moritz\Documents\Uni\Masterarbeit\YETI_MA\Masterarbeit\data\shapefiles\GI_5\Gletscher_AOIs.shp",
    lake_shp = r"C:\Users\Moritz\Documents\Uni\Masterarbeit\YETI_MA\Masterarbeit\data\shapefiles\Seen_AOIs.shp",
    forest_mask_path = r"C:\Users\Moritz\Documents\Uni\Masterarbeit\YETI_MA\Masterarbeit\data\ses_topo_22_forest_mask_domain_epsg32632.tif"
    )

Image shape at 20 m resolution: (1360, 1535) pixels
Total number of results: 47
Available image dates: [datetime.datetime(2022, 10, 3, 0, 0), datetime.datetime(2022, 10, 5, 0, 0), datetime.datetime(2022, 10, 18, 0, 0), datetime.datetime(2022, 10, 25, 0, 0), datetime.datetime(2022, 10, 28, 0, 0), datetime.datetime(2022, 10, 30, 0, 0), datetime.datetime(2022, 11, 14, 0, 0), datetime.datetime(2022, 11, 19, 0, 0), datetime.datetime(2022, 11, 27, 0, 0), datetime.datetime(2022, 12, 17, 0, 0), datetime.datetime(2023, 1, 1, 0, 0), datetime.datetime(2023, 1, 6, 0, 0), datetime.datetime(2023, 1, 16, 0, 0), datetime.datetime(2023, 1, 18, 0, 0), datetime.datetime(2023, 1, 26, 0, 0), datetime.datetime(2023, 1, 31, 0, 0), datetime.datetime(2023, 2, 7, 0, 0), datetime.datetime(2023, 2, 10, 0, 0), datetime.datetime(2023, 2, 12, 0, 0), datetime.datetime(2023, 2, 15, 0, 0), datetime.datetime(2023, 2, 20, 0, 0), datetime.datetime(2023, 3, 2, 0, 0), datetime.datetime(2023, 3, 7, 0, 0), datetime.datetime(2

In [7]:
# Example use for Kuehtai 23-24
process_fsc_data(
    coords= coords_kuehtai,
    resolution=20,
    selected_dates= selected_dates_kuehtai_23_24,
    output_folder=r"C:\Users\Moritz\Documents\Uni\Masterarbeit\YETI_MA\Masterarbeit\data\FSC_data\Kuehtai_23-24",
    time_interval=("2022-10-01", "2023-08-31"),
    eval_fsc_scl_mask=eval_fsc_scl_mask,
    glacier_shp = r"C:\Users\Moritz\Documents\Uni\Masterarbeit\YETI_MA\Masterarbeit\data\shapefiles\GI_5\Gletscher_AOIs.shp",
    lake_shp = r"C:\Users\Moritz\Documents\Uni\Masterarbeit\YETI_MA\Masterarbeit\data\shapefiles\Seen_AOIs.shp",
    forest_mask_path = r"C:\Users\Moritz\Documents\Uni\Masterarbeit\YETI_MA\Masterarbeit\data\ses_topo_22_forest_mask_domain_epsg32632.tif"
    )

Image shape at 20 m resolution: (1360, 1535) pixels
Total number of results: 47
Available image dates: [datetime.datetime(2022, 10, 3, 0, 0), datetime.datetime(2022, 10, 5, 0, 0), datetime.datetime(2022, 10, 18, 0, 0), datetime.datetime(2022, 10, 25, 0, 0), datetime.datetime(2022, 10, 28, 0, 0), datetime.datetime(2022, 10, 30, 0, 0), datetime.datetime(2022, 11, 14, 0, 0), datetime.datetime(2022, 11, 19, 0, 0), datetime.datetime(2022, 11, 27, 0, 0), datetime.datetime(2022, 12, 17, 0, 0), datetime.datetime(2023, 1, 1, 0, 0), datetime.datetime(2023, 1, 6, 0, 0), datetime.datetime(2023, 1, 16, 0, 0), datetime.datetime(2023, 1, 18, 0, 0), datetime.datetime(2023, 1, 26, 0, 0), datetime.datetime(2023, 1, 31, 0, 0), datetime.datetime(2023, 2, 7, 0, 0), datetime.datetime(2023, 2, 10, 0, 0), datetime.datetime(2023, 2, 12, 0, 0), datetime.datetime(2023, 2, 15, 0, 0), datetime.datetime(2023, 2, 20, 0, 0), datetime.datetime(2023, 3, 2, 0, 0), datetime.datetime(2023, 3, 7, 0, 0), datetime.datetime(2

In [8]:
# Example use for Kaunertal 22-23
process_fsc_data(
    coords= coords_kaunertal,
    resolution=20,
    selected_dates= selected_dates_kaunertal_22_23,
    output_folder=r"C:\Users\Moritz\Documents\Uni\Masterarbeit\YETI_MA\Masterarbeit\data\FSC_data\Kaunertal_22-23",
    time_interval=("2022-10-01", "2023-08-31"),
    eval_fsc_scl_mask=eval_fsc_scl_mask,
    glacier_shp = r"C:\Users\Moritz\Documents\Uni\Masterarbeit\YETI_MA\Masterarbeit\data\shapefiles\GI_5\Gletscher_AOIs.shp",
    lake_shp = r"C:\Users\Moritz\Documents\Uni\Masterarbeit\YETI_MA\Masterarbeit\data\shapefiles\Seen_AOIs.shp",
    forest_mask_path = r"C:\Users\Moritz\Documents\Uni\Masterarbeit\YETI_MA\Masterarbeit\data\ses_topo_22_forest_mask_domain_epsg32632.tif"
    )

Image shape at 20 m resolution: (1400, 1315) pixels
Total number of results: 113
Available image dates: [datetime.datetime(2022, 10, 3, 0, 0), datetime.datetime(2022, 10, 5, 0, 0), datetime.datetime(2022, 10, 8, 0, 0), datetime.datetime(2022, 10, 13, 0, 0), datetime.datetime(2022, 10, 18, 0, 0), datetime.datetime(2022, 10, 25, 0, 0), datetime.datetime(2022, 10, 28, 0, 0), datetime.datetime(2022, 10, 30, 0, 0), datetime.datetime(2022, 11, 2, 0, 0), datetime.datetime(2022, 11, 14, 0, 0), datetime.datetime(2022, 11, 17, 0, 0), datetime.datetime(2022, 11, 19, 0, 0), datetime.datetime(2022, 11, 24, 0, 0), datetime.datetime(2022, 11, 27, 0, 0), datetime.datetime(2022, 12, 7, 0, 0), datetime.datetime(2022, 12, 17, 0, 0), datetime.datetime(2022, 12, 24, 0, 0), datetime.datetime(2022, 12, 27, 0, 0), datetime.datetime(2022, 12, 29, 0, 0), datetime.datetime(2023, 1, 1, 0, 0), datetime.datetime(2023, 1, 6, 0, 0), datetime.datetime(2023, 1, 16, 0, 0), datetime.datetime(2023, 1, 18, 0, 0), datetime.

In [9]:
# Example use for Kaunertal 23-24
process_fsc_data(
    coords= coords_kaunertal,
    resolution=20,
    selected_dates= selected_dates_kaunertal_23_24,
    output_folder=r"C:\Users\Moritz\Documents\Uni\Masterarbeit\YETI_MA\Masterarbeit\data\FSC_data\Kaunertal_23-24",
    time_interval=("2022-10-01", "2023-08-31"),
    eval_fsc_scl_mask=eval_fsc_scl_mask,
    glacier_shp = r"C:\Users\Moritz\Documents\Uni\Masterarbeit\YETI_MA\Masterarbeit\data\shapefiles\GI_5\Gletscher_AOIs.shp",
    lake_shp = r"C:\Users\Moritz\Documents\Uni\Masterarbeit\YETI_MA\Masterarbeit\data\shapefiles\Seen_AOIs.shp",
    forest_mask_path = r"C:\Users\Moritz\Documents\Uni\Masterarbeit\YETI_MA\Masterarbeit\data\ses_topo_22_forest_mask_domain_epsg32632.tif"
    )

Image shape at 20 m resolution: (1400, 1315) pixels
Total number of results: 113
Available image dates: [datetime.datetime(2022, 10, 3, 0, 0), datetime.datetime(2022, 10, 5, 0, 0), datetime.datetime(2022, 10, 8, 0, 0), datetime.datetime(2022, 10, 13, 0, 0), datetime.datetime(2022, 10, 18, 0, 0), datetime.datetime(2022, 10, 25, 0, 0), datetime.datetime(2022, 10, 28, 0, 0), datetime.datetime(2022, 10, 30, 0, 0), datetime.datetime(2022, 11, 2, 0, 0), datetime.datetime(2022, 11, 14, 0, 0), datetime.datetime(2022, 11, 17, 0, 0), datetime.datetime(2022, 11, 19, 0, 0), datetime.datetime(2022, 11, 24, 0, 0), datetime.datetime(2022, 11, 27, 0, 0), datetime.datetime(2022, 12, 7, 0, 0), datetime.datetime(2022, 12, 17, 0, 0), datetime.datetime(2022, 12, 24, 0, 0), datetime.datetime(2022, 12, 27, 0, 0), datetime.datetime(2022, 12, 29, 0, 0), datetime.datetime(2023, 1, 1, 0, 0), datetime.datetime(2023, 1, 6, 0, 0), datetime.datetime(2023, 1, 16, 0, 0), datetime.datetime(2023, 1, 18, 0, 0), datetime.